### Assignment 6: Spark Data Processing using PySpark

##### Objective
The goal is to implement a basic end-to-end PySpark pipeline on local mode. This covers schema definition, standard transformations (select, filter, withColumn), handling missing data, and saving the outputs in CSV and Parquet formats while exploring performance concepts like Lazy Evaluation, DAG, and shuffles.

##### Spark Architecture

Apache Spark follows a master-worker architecture.

- **Driver Program:** Creates the SparkSession, converts the code into tasks, and coordinates execution.
- **Cluster Manager:** Allocates resources to Spark applications.
- **Executors:** Execute tasks and process data in parallel.

In this assignment, Spark runs in **Local Mode**, where the Driver and Executors execute on the same machine.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [2]:
spark = (
    SparkSession.builder
    .appName("Spark Assignment - Celebal")
    .master("local[*]")
    .getOrCreate()
)
print("Spark Session Created Successfully")

Spark Session Created Successfully


##### Lazy Evaluation

Apache Spark uses Lazy Evaluation.

Transformations such as `select()`, `filter()`, and `withColumn()` are not executed immediately. Instead, Spark creates a Directed Acyclic Graph (DAG) that records all transformations.

Execution starts only when an Action such as `show()`, `count()`, or `write()` is called. This optimization minimizes unnecessary computations and improves performance.

##### Reading CSV File with Proper Schema

The dataset is loaded into a Spark DataFrame using a predefined schema. Defining the schema explicitly ensures correct data types, improves performance, and avoids relying on automatic schema inference.

In [3]:
schema = StructType([
    StructField("Row ID", IntegerType(), True),
    StructField("Order ID", StringType(), True),
    StructField("Order Date", StringType(), True),
    StructField("Ship Date", StringType(), True),
    StructField("Ship Mode", StringType(), True),
    StructField("Customer ID", StringType(), True),
    StructField("Customer Name", StringType(), True),
    StructField("Segment", StringType(), True),
    StructField("Country", StringType(), True),
    StructField("City", StringType(), True),
    StructField("State", StringType(), True),
    StructField("Postal Code", IntegerType(), True),
    StructField("Region", StringType(), True),
    StructField("Product ID", StringType(), True),
    StructField("Category", StringType(), True),
    StructField("Sub-Category", StringType(), True),
    StructField("Product Name", StringType(), True),
    StructField("Sales", DoubleType(), True),
    StructField("Quantity", IntegerType(), True),
    StructField("Discount", DoubleType(), True),
    StructField("Profit", DoubleType(), True)
])

In [4]:
df = spark.read.csv(
    "Sample - Superstore.csv",
    header=True,
    schema=schema
)

In [5]:
df.printSchema()

root
 |-- Row ID: integer (nullable = true)
 |-- Order ID: string (nullable = true)
 |-- Order Date: string (nullable = true)
 |-- Ship Date: string (nullable = true)
 |-- Ship Mode: string (nullable = true)
 |-- Customer ID: string (nullable = true)
 |-- Customer Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postal Code: integer (nullable = true)
 |-- Region: string (nullable = true)
 |-- Product ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub-Category: string (nullable = true)
 |-- Product Name: string (nullable = true)
 |-- Sales: double (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- Discount: double (nullable = true)
 |-- Profit: double (nullable = true)



In [6]:
df.show(5, truncate=False)

+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+-----------------------------------------------------------+--------+--------+--------+--------+
|Row ID|Order ID      |Order Date|Ship Date |Ship Mode     |Customer ID|Customer Name  |Segment  |Country      |City           |State     |Postal Code|Region|Product ID     |Category       |Sub-Category|Product Name                                               |Sales   |Quantity|Discount|Profit  |
+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+-----------------------------------------------------------+--------+--------+--------+--------+
|1     |CA-2016-152156|11/8/2016 |11/11/2016|Second Class  |CG-12520   |Claire Gute    |Consumer |Un

##### Exploring the Dataset

After loading the dataset, the schema and sample records are displayed to verify that the data has been read correctly and the predefined schema has been applied successfully.

In [7]:
print("Number of Rows:", df.count())
print("Number of Columns:", len(df.columns))

Number of Rows: 9994
Number of Columns: 21


In [8]:
df.columns

['Row ID',
 'Order ID',
 'Order Date',
 'Ship Date',
 'Ship Mode',
 'Customer ID',
 'Customer Name',
 'Segment',
 'Country',
 'City',
 'State',
 'Postal Code',
 'Region',
 'Product ID',
 'Category',
 'Sub-Category',
 'Product Name',
 'Sales',
 'Quantity',
 'Discount',
 'Profit']

##### Selecting Required Columns

The `select()` transformation is used to retrieve only the required columns from the DataFrame. This reduces unnecessary data processing and improves readability.

In [9]:
selected_df = df.select(
    "Customer Name",
    "Category",
    "Sales",
    "Profit"
)

selected_df.show(5, truncate=False)

+---------------+---------------+--------+--------+
|Customer Name  |Category       |Sales   |Profit  |
+---------------+---------------+--------+--------+
|Claire Gute    |Furniture      |261.96  |41.9136 |
|Claire Gute    |Furniture      |731.94  |219.582 |
|Darrin Van Huff|Office Supplies|14.62   |6.8714  |
|Sean O'Donnell |Furniture      |957.5775|-383.031|
|Sean O'Donnell |Office Supplies|22.368  |2.5164  |
+---------------+---------------+--------+--------+
only showing top 5 rows


##### Filtering Data

The `filter()` transformation is used to retrieve only those records that satisfy a given condition.

In [10]:
high_sales = df.filter(col("Sales") > 500)

high_sales.show(5, truncate=False)

+------+--------------+----------+----------+--------------+-----------+---------------+--------+-------------+---------------+----------+-----------+------+---------------+----------+------------+-----------------------------------------------------------+--------+--------+--------+--------+
|Row ID|Order ID      |Order Date|Ship Date |Ship Mode     |Customer ID|Customer Name  |Segment |Country      |City           |State     |Postal Code|Region|Product ID     |Category  |Sub-Category|Product Name                                               |Sales   |Quantity|Discount|Profit  |
+------+--------------+----------+----------+--------------+-----------+---------------+--------+-------------+---------------+----------+-----------+------+---------------+----------+------------+-----------------------------------------------------------+--------+--------+--------+--------+
|2     |CA-2016-152156|11/8/2016 |11/11/2016|Second Class  |CG-12520   |Claire Gute    |Consumer|United States|Henders

In [11]:
technology_orders = df.filter(col("Category") == "Technology")

technology_orders.show(5, truncate=False)

+------+--------------+----------+----------+--------------+-----------+------------------+---------+-------------+-------------+----------+-----------+-------+---------------+----------+------------+------------------------------------------------+--------+--------+--------+--------+
|Row ID|Order ID      |Order Date|Ship Date |Ship Mode     |Customer ID|Customer Name     |Segment  |Country      |City         |State     |Postal Code|Region |Product ID     |Category  |Sub-Category|Product Name                                    |Sales   |Quantity|Discount|Profit  |
+------+--------------+----------+----------+--------------+-----------+------------------+---------+-------------+-------------+----------+-----------+-------+---------------+----------+------------+------------------------------------------------+--------+--------+--------+--------+
|8     |CA-2014-115812|6/9/2014  |6/14/2014 |Standard Class|BH-11710   |Brosina Hoffman   |Consumer |United States|Los Angeles  |California|90

##### Transformations and Actions

Spark transformations create a new DataFrame without executing immediately. Execution begins only when an action such as `show()` or `count()` is performed.

In [12]:
print("Total Orders:", df.count())
print("Technology Orders:", technology_orders.count())

Total Orders: 9994
Technology Orders: 1847


##### Modifying the DataFrame

DataFrames can be modified by renaming columns, changing data types, and creating new columns. These operations help prepare the data for further analysis.

In [13]:
modified_df = df.withColumnRenamed("Customer Name", "Customer_Name") \
                .withColumnRenamed("Product Name", "Product_Name")

modified_df.show(5)

+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|  Customer_Name|  Segment|      Country|           City|     State|Postal Code|Region|     Product ID|       Category|Sub-Category|        Product_Name|   Sales|Quantity|Discount|  Profit|
+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|     1|CA-2016-152156| 11/8/2016|11/11/2016|  Second Class|   CG-12520|    Claire Gute| Consumer|United States|      Henderson|  Kentucky|      42420| South|FUR-BO-10001798|      Furniture|   Bookcases|Bush Somerset 

In [14]:
modified_df = modified_df.withColumn(
    "Quantity",
    col("Quantity").cast("double")
)

modified_df.printSchema()

root
 |-- Row ID: integer (nullable = true)
 |-- Order ID: string (nullable = true)
 |-- Order Date: string (nullable = true)
 |-- Ship Date: string (nullable = true)
 |-- Ship Mode: string (nullable = true)
 |-- Customer ID: string (nullable = true)
 |-- Customer_Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postal Code: integer (nullable = true)
 |-- Region: string (nullable = true)
 |-- Product ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub-Category: string (nullable = true)
 |-- Product_Name: string (nullable = true)
 |-- Sales: double (nullable = true)
 |-- Quantity: double (nullable = true)
 |-- Discount: double (nullable = true)
 |-- Profit: double (nullable = true)



In [26]:
modified_df = modified_df.withColumn(
    "Order Date",
    to_date(col("Order Date"), "MM/dd/yyyy")
).withColumn(
    "Ship Date",
    to_date(col("Ship Date"), "MM/dd/yyyy")
)

modified_df.printSchema()

root
 |-- Row ID: integer (nullable = true)
 |-- Order ID: string (nullable = true)
 |-- Order Date: date (nullable = true)
 |-- Ship Date: date (nullable = true)
 |-- Ship Mode: string (nullable = true)
 |-- Customer ID: string (nullable = true)
 |-- Customer_Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postal Code: integer (nullable = true)
 |-- Region: string (nullable = true)
 |-- Product ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub-Category: string (nullable = true)
 |-- Product_Name: string (nullable = true)
 |-- Sales: double (nullable = true)
 |-- Quantity: double (nullable = true)
 |-- Discount: double (nullable = true)
 |-- Profit: double (nullable = true)



In [15]:
modified_df = modified_df.withColumn(
    "Net Sales",
    col("Sales") - (col("Sales") * col("Discount"))
)

modified_df.select(
    "Sales",
    "Discount",
    "Net Sales"
).show(5)

+--------+--------+------------------+
|   Sales|Discount|         Net Sales|
+--------+--------+------------------+
|  261.96|     0.0|            261.96|
|  731.94|     0.0|            731.94|
|   14.62|     0.0|             14.62|
|957.5775|    0.45|        526.667625|
|  22.368|     0.2|17.894399999999997|
+--------+--------+------------------+
only showing top 5 rows


In [16]:
from pyspark.sql.functions import expr

modified_df = modified_df.withColumn(
    "Order Date",
    expr("try_to_timestamp(`Order Date`, 'M/d/yyyy')").cast("date")
).withColumn(
    "Ship Date",
    expr("try_to_timestamp(`Ship Date`, 'M/d/yyyy')").cast("date")
)

modified_df.printSchema()

root
 |-- Row ID: integer (nullable = true)
 |-- Order ID: string (nullable = true)
 |-- Order Date: date (nullable = true)
 |-- Ship Date: date (nullable = true)
 |-- Ship Mode: string (nullable = true)
 |-- Customer ID: string (nullable = true)
 |-- Customer_Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postal Code: integer (nullable = true)
 |-- Region: string (nullable = true)
 |-- Product ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub-Category: string (nullable = true)
 |-- Product_Name: string (nullable = true)
 |-- Sales: double (nullable = true)
 |-- Quantity: double (nullable = true)
 |-- Discount: double (nullable = true)
 |-- Profit: double (nullable = true)
 |-- Net Sales: double (nullable = true)



In [17]:
modified_df = modified_df.withColumn(
    "Net Sales",
    round(col("Sales") * (1 - col("Discount")), 2)
)

modified_df.select(
    "Sales",
    "Discount",
    "Net Sales"
).show(5)

+--------+--------+---------+
|   Sales|Discount|Net Sales|
+--------+--------+---------+
|  261.96|     0.0|   261.96|
|  731.94|     0.0|   731.94|
|   14.62|     0.0|    14.62|
|957.5775|    0.45|   526.67|
|  22.368|     0.2|    17.89|
+--------+--------+---------+
only showing top 5 rows


##### DataFrame Modification Summary

The DataFrame was modified by renaming columns, changing the data type of a column, and creating a new calculated column. These transformations prepare the data for further analysis.

##### Handling Null Values

Before analysis, it is important to identify and handle missing values to improve data quality.

In [18]:
modified_df.select([
    sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in modified_df.columns
]).show()

+------+--------+----------+---------+---------+-----------+-------------+-------+-------+----+-----+-----------+------+----------+--------+------------+------------+-----+--------+--------+------+---------+
|Row ID|Order ID|Order Date|Ship Date|Ship Mode|Customer ID|Customer_Name|Segment|Country|City|State|Postal Code|Region|Product ID|Category|Sub-Category|Product_Name|Sales|Quantity|Discount|Profit|Net Sales|
+------+--------+----------+---------+---------+-----------+-------------+-------+-------+----+-----+-----------+------+----------+--------+------------+------------+-----+--------+--------+------+---------+
|     0|       0|         0|        0|        0|          0|            0|      0|      0|   0|    0|          0|     0|         0|       0|           0|           0|  300|     300|      11|     0|      300|
+------+--------+----------+---------+---------+-----------+-------------+-------+-------+----+-----+-----------+------+----------+--------+------------+------------+--

In [19]:
clean_df = modified_df.dropna()

print("Rows before removing null values:", modified_df.count())
print("Rows after removing null values:", clean_df.count())

Rows before removing null values: 9994
Rows after removing null values: 9694


##### Wide Transformation

Grouping data requires Spark to shuffle records across partitions. This is an example of a wide transformation.

In [20]:
category_sales = clean_df.groupBy("Category").agg(
    round(sum("Sales"), 2).alias("Total Sales"),
    round(avg("Profit"), 2).alias("Average Profit")
)

category_sales.show()

+---------------+-----------+--------------+
|       Category|Total Sales|Average Profit|
+---------------+-----------+--------------+
|Office Supplies|  703502.93|         20.84|
|      Furniture|  733046.86|          8.19|
|     Technology|  835900.07|         79.06|
+---------------+-----------+--------------+



In [21]:
category_sales.explain(True)

== Parsed Logical Plan ==
'Aggregate ['Category], ['Category, 'round('sum('Sales), 2) AS Total Sales#763, 'round('avg('Profit), 2) AS Average Profit#764]
+- Filter atleastnnonnulls(22, Row ID#0, Order ID#1, Order Date#471, Ship Date#472, Ship Mode#4, Customer ID#5, Customer_Name#370, Segment#7, Country#8, City#9, State#10, Postal Code#11, Region#12, Product ID#13, Category#14, Sub-Category#15, Product_Name#371, Sales#17, Quantity#457, Discount#19, Profit#20, Net Sales#473)
   +- Project [Row ID#0, Order ID#1, Order Date#471, Ship Date#472, Ship Mode#4, Customer ID#5, Customer_Name#370, Segment#7, Country#8, City#9, State#10, Postal Code#11, Region#12, Product ID#13, Category#14, Sub-Category#15, Product_Name#371, Sales#17, Quantity#457, Discount#19, Profit#20, round((Sales#17 * (cast(1 as double) - Discount#19)), 2) AS Net Sales#473]
      +- Project [Row ID#0, Order ID#1, Order Date#471, cast(try_to_timestamp(Ship Date#3, Some(M/d/yyyy), TimestampType, Some(Asia/Calcutta), false) as d

##### Data Export Pipeline (Local Environment Execution Note)
*Architectural Note:* In a standard production environment, native distributed writing via `df.write.parquet()` is preferred. However, due to local Windows environment constraints (lack of native Hadoop binaries / `winutils.exe` setup causing write errors on file overwrite), the dataset is converted locally to handle file system output gracefully without failing the session.

In [26]:
clean_df.toPandas().to_csv("output.csv", index=False)

In [30]:
clean_df.toPandas().to_parquet("output.parquet")

##### Reading Parquet File

In [33]:
import pandas as pd

df = pd.read_parquet("output.parquet")
df.head()

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer_Name,Segment,Country,City,...,Region,Product ID,Category,Sub-Category,Product_Name,Sales,Quantity,Discount,Profit,Net Sales
0,1,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2.0,0.00,41.9136,261.96
1,2,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3.0,0.00,219.5820,731.94
2,3,CA-2016-138688,2016-06-12,2016-06-16,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2.0,0.00,6.8714,14.62
3,4,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5.0,0.45,-383.0310,526.67
4,5,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2.0,0.20,2.5164,17.89


##### CSV vs Parquet

CSV is a plain text file format that is easy to read but requires more storage and slower processing.

Parquet is a columnar storage format that provides better compression, faster query performance, and supports Predicate Pushdown, making it suitable for big data processing.

##### Performance Insights

- Manual schema improves performance by avoiding schema inference.
- Lazy Evaluation executes transformations only when an action is called.
- `groupBy()` is a wide transformation that involves shuffle.
- Parquet provides better performance than CSV because it stores data in a columnar format.
- `show()` was used instead of `collect()` to avoid bringing the entire dataset into driver memory.

In [34]:
spark.stop()